# 5.7 — FastTD3 directional walk from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/tutorial/fasttd3-directional-scratch/tutorials/5.7_FastTD3_Directional_From_Scratch.ipynb)

One cell: install MyoSuite, then FastTD3 from scratch on `myoFullBodyDirectional-v0` (528 → 354). No MuscleMimic teacher, no Hugging Face token.

Keep `SMOKE = True` on a **CPU** runtime until it prints `ok`. Then Runtime → GPU, set `SMOKE = False`, run the cell again.

In [ ]:
from __future__ import annotations

SMOKE = True  # CPU plumbing. False = train (needs GPU).
TRAIN_STEPS = 200_000
N_ENVS = 8
BATCH = 1024

import os
import subprocess
import sys
from pathlib import Path

GIT_URL = "https://github.com/MyoHub/myosuite.git"
GIT_REF = "tutorial/fasttd3-directional-scratch"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "libegl1", "libgles2", "libosmesa6"],
        check=False,
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "imageio[ffmpeg]", "mediapy"]
    )
    dest = Path("/content/myosuite")
    if dest.exists():
        subprocess.check_call(["git", "-C", str(dest), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(dest), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    else:
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(dest)]
        )
    os.chdir(dest)
    if str(dest) not in sys.path:
        sys.path.insert(0, str(dest))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{dest}[musclemimic]"]
    )
    repo = dest
else:
    here = Path.cwd().resolve()
    repo = next((d for d in (here, *here.parents) if (d / "myosuite" / "__init__.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Run this notebook from a myosuite clone, or open it in Colab.")
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

helper = repo / "tutorials" / "files" / "5.7"
if not (helper / "fasttd3_directional.py").is_file():
    raise FileNotFoundError(f"missing {helper / 'fasttd3_directional.py'} — wrong git ref {GIT_REF!r}")
if str(helper) not in sys.path:
    sys.path.insert(0, str(helper))

from fasttd3_directional import eval_dirs, get_device, render_east, train  # noqa: E402

device = get_device(smoke=SMOKE)
if not SMOKE and device.type == "cpu":
    raise SystemExit("No GPU. Runtime → T4/A100, or keep SMOKE=True.")

steps = 400 if SMOKE else TRAIN_STEPS
n_envs = 1 if SMOKE else N_ENVS
batch = 64 if SMOKE else BATCH
out = Path("runs/fasttd3_scratch")
print(f"IN_COLAB={IN_COLAB} SMOKE={SMOKE} device={device} steps={steps} envs={n_envs}")

actor = train(steps=steps, n_envs=n_envs, batch=batch, device=device, out=out, smoke=SMOKE)
summary = eval_dirs(actor, device, labels=["E", "N"] if SMOKE else None, steps=40 if SMOKE else 120)
print("summary", {k: round(v, 3) for k, v in summary.items()})
if SMOKE:
    print("ok — CPU loop works. Set SMOKE=False on a GPU runtime to train.")
else:
    mp4 = render_east(actor, device, out / "east.mp4")
    try:
        from IPython.display import Video, display
        display(Video(str(mp4), embed=True, width=420))
    except Exception:
        print(mp4)
